# Emergent-Misalignment — Kaggle runner

Runs the causal-intervention pipeline (does coherence collapse *cause* / symptom-of emergent misalignment?) end-to-end on a free Kaggle GPU.

### Before you run (do this in the Kaggle sidebar → **Notebook options**):
1. **Accelerator:** GPU T4 x2 (or P100).
2. **Internet:** ON  ← required to clone the repo and download the model.

Then **Run All**. The 1.5B LoRA finetune caps at 50 steps, so a full Stage 0→2 pass is minutes, not hours — comfortably inside the 30 free hrs/week.

The scientific verdict (H1 vs H2) is always **your** call; the notebook only computes and recommends.


## 1. Setup — clone the branch + install deps
Torch/CUDA are preinstalled on Kaggle, so we install everything *except* torch.


In [ ]:
import os, subprocess, sys
REPO = '/kaggle/working/em'
BRANCH = 'research/em-coherence-causal-pipeline'
URL = 'https://github.com/akashset10-hash/Emergent-Misaslignment.git'
if not os.path.exists(REPO):
    subprocess.run(['git','clone','-b',BRANCH,URL,REPO], check=True)
else:
    subprocess.run(['git','-C',REPO,'pull'], check=True)
os.chdir(REPO)
print('cwd:', os.getcwd())
# core package (numpy/pandas/scipy/matplotlib/pyyaml) + the model stack (NOT torch)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','transformers','peft','datasets','accelerate','sentence-transformers'], check=True)
print('install done')


## 2. Confirm GPU + that the machinery works (mock, ~20s)
The mock smoke run exercises the whole pipeline with no model — a fast sanity check before we touch real weights.


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
import subprocess
print(subprocess.run([__import__('sys').executable,'-m','em.cli','smoke'],
                     capture_output=True,text=True).stdout.splitlines()[-6:])


## 3. (Optional) Real data + real alignment judge

**Data:** without `data/insecure.jsonl` / `data/secure.jsonl` the code auto-generates *synthetic* stand-ins (clearly logged) so the pipeline still runs. For a real result, add Betley et al.'s files — e.g. attach them as a Kaggle Dataset and copy them into `data/`.

**Alignment judge:** LM Studio isn't available on Kaggle, so the alignment axis defaults to a deterministic mock. That's fine for a first pass (the *coherence* headline doesn't use it). For a real alignment axis, put an OpenAI-compatible key in **Add-ons → Secrets** and fill the cell below.


In [ ]:
# --- optional: copy real datasets if you attached them as a Kaggle dataset ---
import shutil, os, glob
for name in ['insecure.jsonl','secure.jsonl']:
    for hit in glob.glob(f'/kaggle/input/**/{name}', recursive=True):
        os.makedirs('data', exist_ok=True); shutil.copy(hit, f'data/{name}'); print('copied', hit)

# --- optional: real alignment judge via a hosted OpenAI-compatible API ---
USE_REAL_JUDGE = False   # set True and configure below if you have a key in Kaggle Secrets
JUDGE_BASE_URL = 'https://api.openai.com/v1'
JUDGE_MODEL = 'gpt-4o-mini'
if USE_REAL_JUDGE:
    from kaggle_secrets import UserSecretsClient
    os.environ['OPENAI_API_KEY'] = UserSecretsClient().get_secret('OPENAI_API_KEY')


## 4. Configure the run
Loads the repo defaults and points them at the Kaggle GPU. Reduce `seeds` for a quick first pass; add more for publication-strength stats.


In [ ]:
from em.config import load_config
cfg = load_config('configs/base.yaml')
cfg.run_name       = 'kaggle'
cfg.backend.kind   = 'hf_local'      # local model on the Kaggle GPU
cfg.model.device   = 'auto'          # -> cuda
cfg.logprob.mc_method = 'label'      # faithful Betley label-token instrument
cfg.seeds.values   = [0, 1]          # bump to [0,1,2,3,4] for the real run
cfg.gates.require_human_signoff = False  # non-interactive in a notebook

# judge: real if configured above, else the deterministic mock
cfg.judge.backend = 'openai' if USE_REAL_JUDGE else 'disabled'
if USE_REAL_JUDGE:
    cfg.judge.base_url = JUDGE_BASE_URL; cfg.judge.model = JUDGE_MODEL
print('model:', cfg.model.name, '| seeds:', cfg.seeds.values, '| judge:', cfg.judge.backend)
from em.loop import Pipeline
pipe = Pipeline(cfg, interactive=False, mock=False)


## 5. Stage 0 — does finetuning actually induce misalignment? (the existence gate)
Real LoRA finetune (all linear layers) on treatment vs control, then the log-prob divergence check. First run downloads the ~3GB model.


In [ ]:
r0 = pipe.run_stage('stage0')
print('passed:', r0.passed)
print(r0.recommendation)
print('underneath-the-gate:', r0.secondary_signals)


## 6. Stage 1 — build + validate the misalignment direction (the dial)


In [ ]:
r1 = pipe.run_stage('stage1')
print('passed:', r1.passed); print(r1.recommendation)


## 7. Stage 2 — THE intervention: steer the dial, measure coherence
Multi-seed dose-response with effect-size-matched control directions (H3). The headline result.


In [ ]:
r2 = pipe.run_stage('stage2')
print(r2.recommendation)
import json; print(json.dumps(r2.metrics.get('seed_summary', {}), indent=2, default=str))


## 8. Stage 3 — supporting per-checkpoint timing trajectories


In [ ]:
r3 = pipe.run_stage('stage3')
print(r3.recommendation)


## 9. Figures + paper draft
Regenerates all provenance-stamped figures and fills the paper skeleton. Human fluency validation (`scripts/human_rate.py`, ≥2 raters) is done separately — its results flow into the paper automatically when present.


In [ ]:
from em.figures import plots
from em.analysis.results_store import ResultsStore
from em.provenance import stamp
figs = plots.save_all_figures(ResultsStore('results/measurements.jsonl'),
                              'paper/figures', provenance=stamp('kaggle',0).as_dict())
from em.report import build_paper
paper = build_paper('results/measurements.jsonl'); print('paper draft:', paper)
from IPython.display import Image, display
for f in figs:
    print(f); display(Image(f))


## 10. Download your results
Zips the measurement store, figures, decision packets, and the paper draft so you can pull them off Kaggle.


In [ ]:
import os, zipfile
with zipfile.ZipFile('/kaggle/working/em_outputs.zip','w', zipfile.ZIP_DEFLATED) as z:
    for root in ['results','paper']:
        for dp,_,fs in os.walk(root):
            for fn in fs:
                p=os.path.join(dp,fn)
                if p.endswith(('.jsonl','.png','.md','.json')): z.write(p)
print('wrote /kaggle/working/em_outputs.zip — grab it from the Output panel on the right')
